In [ ]:
import osmnx as ox
from osmnx.features import features_from_bbox
import geopandas as gpd
from src.feature_building_utils import *
from src.geometric_utils import *
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import toml

In [ ]:
df = pd.read_parquet('data/processed_data/S3-approx-coordinates.parquet')

In [ ]:
docs = toml.load("documentation/feature_docs.toml")

In [ ]:
tags_public_transport = {"public_transport": True}
gdf_public_transport = features_from_bbox(BBOX, tags_public_transport)

In [ ]:
feature_name = "close2public_transport_15m"

df[feature_name] = df.apply(
    lambda row: is_close_to(
        features=gdf_public_transport,
        point=Point(row.x, row.y),
        threshold=15
    ),
    axis=1
).astype(int)

# add a new metadata field
docs.setdefault(feature_name, {})
docs[feature_name]["description"] = "'Dummy' variable indicating whether the point is less than 15m away from a public transport infrastructure."
docs[feature_name]["type"] = "boolean"
docs[feature_name]["range"] = {int(df[feature_name].min()), int(df[feature_name].max())}
docs[feature_name]["created_on"] = "osmnx_public_transport.ipynb"
docs[feature_name]["source"] = "OSM"
docs[feature_name]["source_url"] = "https://www.openstreetmap.org/"

In [ ]:
feature_name = "num_public_transport_50"

df[feature_name] = df.apply(
    lambda row: count_nearby(
        features=gdf_public_transport,
        point=Point(row.x, row.y),
        threshold=50
    ),
    axis=1
).astype(int)

# add a new metadata field
docs.setdefault(feature_name, {})
docs[feature_name]["description"] = "Number of public transport within 50m of the point"
docs[feature_name]["type"] = "integer"
docs[feature_name]["range"] = {int(df[feature_name].min()), int(df[feature_name].max())}
docs[feature_name]["created_on"] = "osmnx_public_transport.ipynb"
docs[feature_name]["source"] = "OSM"
docs[feature_name]["source_url"] = "https://www.openstreetmap.org/"

In [ ]:
feature_name = "num_public_transport_100"

df[feature_name] = df.apply(
    lambda row: count_nearby(
        features=gdf_public_transport,
        point=Point(row.x, row.y),
        threshold=100
    ),
    axis=1
).astype(int)

# add a new metadata field
docs.setdefault(feature_name, {})
docs[feature_name]["description"] = "Number of public transport within 100m of the point"
docs[feature_name]["type"] = "integer"
docs[feature_name]["range"] = {int(df[feature_name].min()), int(df[feature_name].max())}
docs[feature_name]["created_on"] = "osmnx_public_transport.ipynb"
docs[feature_name]["source"] = "OSM"
docs[feature_name]["source_url"] = "https://www.openstreetmap.org/"

In [ ]:
with open("documentation/feature_docs.toml", "w") as f:
    toml.dump(docs, f)

In [ ]:
df.to_parquet('data/processed_data/S3-approx-coordinates.parquet')